# 01 — Pull market data and cache it

**Goal:** download the history for every ticker used by our monitored
relationships and save it to `data/prices.csv`.

### Why cache to a file instead of pulling every time?

1. Bloomberg data limits: every BDH call counts against your monthly quota.
   Pulling once a day and reusing the file is polite to your quota.
2. Speed: the detection notebook re-runs instantly from the CSV.
3. Reproducibility: you can re-run analysis on the exact same data.

### The relationships we monitor

They live in `pairs_config.py` (open it — it is just a Python list).
Each pair says which two tickers to compare, how to turn their levels
into daily moves, and what sign the co-movement "should" have.

In [1]:
import datetime as dt
import os

import bbg
from pairs_config import PAIRS, all_tickers

for pair in PAIRS:
    sign = {1: "+", -1: "-", 0: "auto"}[pair["expected_sign"]]
    print(f"{pair['name']:<28} expected sign: {sign:<5} "
          f"({pair['ticker_a']}  vs  {pair['ticker_b']})")

Equity vs CDX HY             expected sign: -     (SPX Index  vs  CDX HY CDSI GEN 5Y SPRD Corp)
Equity vs CDX IG             expected sign: -     (SPX Index  vs  CDX IG CDSI GEN 5Y Corp)
Equity vs HY cash OAS        expected sign: -     (SPX Index  vs  LF98OAS Index)
EU equity vs Crossover       expected sign: -     (SX5E Index  vs  ITRX XOVER CDSI GEN 5Y Corp)
Equity vs VIX                expected sign: -     (SPX Index  vs  VIX Index)
Rates vol vs Equity vol      expected sign: +     (MOVE Index  vs  VIX Index)
Rates vs Equity              expected sign: auto  (USGG10YR Index  vs  SPX Index)
Rates vs USDJPY              expected sign: +     (USGG10YR Index  vs  USDJPY Curncy)
Bunds vs Treasuries          expected sign: +     (GDBR10 Index  vs  USGG10YR Index)
Oil vs USDCAD                expected sign: -     (CL1 Comdty  vs  USDCAD Curncy)
Oil vs 10y breakevens        expected sign: +     (CL1 Comdty  vs  USGGBE10 Index)
Copper vs AUDUSD             expected sign: +     (HG1 Comdty  

### Pull 3 years of daily history

Three years gives the long-run (1-year) rolling correlation plenty of
data to work with, while staying light on your data quota.

We pull `PX_LAST` for everything. For yield/spread tickers Bloomberg
returns the yield/spread itself as `PX_LAST`, which is what we want.

In [2]:
end = dt.date.today()
start = end - dt.timedelta(days=365 * 3)

tickers = all_tickers()
print(f"Pulling {len(tickers)} tickers from {start} to {end} ...")

prices = bbg.bdh(tickers, "PX_LAST", start, end)
print(f"Got {len(prices)} rows x {len(prices.columns)} columns")

Pulling 30 tickers from 2023-07-10 to 2026-07-09 ...
Got 784 rows x 30 columns


### Clean and save

Two small but important cleaning steps:

1. **Forward-fill** (`ffill`): different markets have different holidays,
   so some tickers have gaps on days others traded. Forward-fill carries
   the last known value forward, which is the standard fix.
2. **Drop leading rows** that are still empty (before a series started).

In [3]:
prices = prices.ffill().dropna(how="any")

os.makedirs("data", exist_ok=True)
prices.to_csv(os.path.join("data", "prices.csv"))
print(f"Saved {len(prices)} clean rows to data/prices.csv")
print(prices.tail(3).round(2))

Saved 784 clean rows to data/prices.csv
            SPX Index  NDX Index  RTY Index  SX5E Index  NKY Index  \
2026-07-07    2852.18   13997.73    1374.95     3747.55   29108.84   
2026-07-08    2912.74   14309.64    1409.06     3774.21   29874.89   
2026-07-09    2947.84   14387.99    1429.06     3809.65   29994.32   

            MXEF Index  VIX Index  MOVE Index  CDX HY CDSI GEN 5Y SPRD Corp  \
2026-07-07      919.81      23.16       53.76                        368.28   
2026-07-08      936.41      21.95       50.96                        360.71   
2026-07-09      934.45      20.33       49.11                        358.77   

            CDX IG CDSI GEN 5Y Corp  ...  EURUSD Curncy  USDJPY Curncy  \
2026-07-07                    29.80  ...           1.23         145.00   
2026-07-08                    27.72  ...           1.24         145.63   
2026-07-09                    25.82  ...           1.23         145.77   

            USDCAD Curncy  AUDUSD Curncy  USDMXN Curncy  CL1 Comd

Done. Next: `02_detect_incoherence` reads this CSV — you never need to
hit Bloomberg again until you want fresh data (just re-run this notebook).